In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "6"
import sys
import json
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
from peft import LoraConfig, get_peft_model
import warnings

# Suppress the specific ambiguous channel warning
warnings.filterwarnings("ignore", message="The channel dimension is ambiguous")

In [2]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
# Adjust these paths to match your actual folder structure
class Config:
    # Model / Training
    MODEL_ID = "openai/clip-vit-base-patch32"
    OUTPUT_DIR = "./clip_coco_lora_checkpoints"
    
    # Data Paths (Update these!)
    # Example: "/path/to/coco/train2017"
    IMAGE_DIR = "../conceptual_captions_data/train" 
    # Example: "/path/to/coco/annotations/captions_train2017.json"
    ANNOTATION_FILE = "../conceptual_captions_data/train.json" 
    
    # Hyperparameters
    BATCH_SIZE = 32       # Lower this if you run out of VRAM (e.g., to 16 or 8)
    LEARNING_RATE = 5e-5  # LoRA typically needs a slightly higher LR than full finetuning
    NUM_EPOCHS = 3        # 3-5 epochs is usually enough for COCO adaptation
    MAX_LENGTH = 77       # CLIP's context length limit
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Config:
    # Model
    MODEL_ID = "openai/clip-vit-base-patch32"
    OUTPUT_DIR = "./clip_lora_checkpoints"
    
    # --- Updated Data Paths ---
    # Based on your snippet: DATA_ROOT = Path("conceptual_captions_data")
    DATA_ROOT = "../conceptual_captions_data/"
    
    # The images are usually stored relative to DATA_ROOT
    IMAGE_DIR = DATA_ROOT 
    
    # The annotations are in a .jsonl file
    ANNOTATION_FILE = os.path.join(DATA_ROOT, "train.jsonl")
    
    # Hyperparameters
    BATCH_SIZE = 64
    LEARNING_RATE = 5e-5
    NUM_EPOCHS = 3
    MAX_LENGTH = 77
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
class ConceptualCaptionsDataset(Dataset):
    def __init__(self, image_dir, annotation_file, processor):
        self.image_dir = image_dir
        self.processor = processor
        self.samples = []
        
        print(f"Loading annotations from: {annotation_file}")
        
        # READ JSONL FILE (Line by Line)
        with open(annotation_file, 'r') as f:
            for line in tqdm(f, desc="Parsing JSONL"):
                try:
                    # Parse each line as a separate JSON object
                    entry = json.loads(line.strip())
                    
                    # Identify keys (CC datasets often vary in key names)
                    # We look for common names for captions and image paths
                    caption = entry.get("caption") or entry.get("text")
                    rel_path = entry.get("filepath") or entry.get("image_path") or entry.get("file_name")
                    
                    if caption and rel_path:
                        self.samples.append({
                            "caption": caption,
                            "image_path": rel_path
                        })
                except json.JSONDecodeError:
                    continue # Skip broken lines
                    
        print(f"Found {len(self.samples)} valid samples.")
        
        if len(self.samples) > 0:
            print("Sample entry:", self.samples[0])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        
        # Construct full path: DATA_ROOT + relative path from jsonl
        # e.g. "conceptual_captions_data" + "train/00001.jpg"
        image_path = os.path.join(self.image_dir, item["image_path"])
        
        try:
            image = Image.open(image_path).convert("RGB")
            
            inputs = self.processor(
                text=[item["caption"]],
                images=image,
                return_tensors="pt",
                padding="max_length",
                truncation=True,
                max_length=Config.MAX_LENGTH
            )
            
            return {
                "pixel_values": inputs["pixel_values"].squeeze(0),
                "input_ids": inputs["input_ids"].squeeze(0),
                "attention_mask": inputs["attention_mask"].squeeze(0)
            }
        except Exception as e:
            # If image fails to load, print error and skip to next index
            # print(f"Error loading {image_path}: {e}")
            return self.__getitem__((idx + 1) % len(self))

In [5]:
def train_one_epoch(model, dataloader, optimizer, epoch_index):
    model.train()
    total_loss = 0
    loop = tqdm(dataloader, desc=f"Epoch {epoch_index+1}/{Config.NUM_EPOCHS}")
    
    for batch in loop:
        batch = {k: v.to(Config.DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        
        # CLIP forward pass
        outputs = model(**batch, return_loss=True)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
        
    return total_loss / len(dataloader)

In [6]:
if __name__ == "__main__":
    # 1. Check Paths
    if not os.path.exists(Config.ANNOTATION_FILE):
        print(f"CRITICAL ERROR: Annotation file not found at {Config.ANNOTATION_FILE}")
    else:
        # 2. Initialize Model & Processor
        print(f"Loading {Config.MODEL_ID}...")
        # Ensure use_fast=True for speed
        processor = CLIPProcessor.from_pretrained(Config.MODEL_ID, use_fast=True)
        base_model = CLIPModel.from_pretrained(Config.MODEL_ID)
        
        # LoRA Config
        peft_config = LoraConfig(
            r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], 
            lora_dropout=0.05, bias="none"
        )
        model = get_peft_model(base_model, peft_config)
        model.to(Config.DEVICE)
        
        # 3. Load Data (With Performance Optimizations)
        dataset = ConceptualCaptionsDataset(Config.IMAGE_DIR, Config.ANNOTATION_FILE, processor)
        
        dataloader = DataLoader(
            dataset, 
            batch_size=Config.BATCH_SIZE, 
            shuffle=True, 
            num_workers=4,         # Number of CPU cores for loading
            pin_memory=True,       # FASTER: Pins memory for faster GPU transfer
            persistent_workers=True # FASTER: Keeps workers alive between epochs
        )
        
        # 4. Train
        optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)
        os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
        
        print("\nStarting Training...")
        for epoch in range(Config.NUM_EPOCHS):
            avg_loss = train_one_epoch(model, dataloader, optimizer, epoch)
            print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")
            
            # Save Adapter
            save_path = os.path.join(Config.OUTPUT_DIR, f"epoch-{epoch+1}")
            model.save_pretrained(save_path)
            print(f"Saved adapter to {save_path}")
            
        print("Done!")

Loading openai/clip-vit-base-patch32...
Loading annotations from: ../conceptual_captions_data/train.jsonl


Parsing JSONL: 3318333it [00:10, 310410.81it/s]


Found 3318333 valid samples.
Sample entry: {'caption': 'a very typical bus station', 'image_path': 'train/00000000.jpg'}

Starting Training...


Epoch 1/3:   3%|▎         | 1629/51849 [08:58<3:56:56,  3.53it/s, loss=0.388]The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension.
Epoch 1/3:   4%|▍         | 2192/51849 [13:04<4:20:24,  3.18it/s, loss=0.253]  /home/usama/anaconda3/envs/analysis/lib/python3.9/site-packages/PIL/Image.py:3406: DecompressionBombWarning: Image size (93950400 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
Epoch 1/3:   9%|▊         | 4532/51849 [25:50<3:04:17,  4.28it/s, loss=0.19]  The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension.
Epoch 1/3:  11%|█▏        | 5863/51849 [33:08<6:29:27,  1.97it/s, loss=0.148] /home/usama/anaconda3/envs/analysis

Epoch 1 Loss: 0.1926
Saved adapter to ./clip_lora_checkpoints/epoch-1


Epoch 2/3:   1%|          | 535/51849 [04:25<4:22:51,  3.25it/s, loss=0.105]  The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension.
Epoch 2/3:   8%|▊         | 3986/51849 [1:24:44<3:29:25,  3.81it/s, loss=0.321] The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension.
Epoch 2/3:   9%|▉         | 4711/51849 [1:43:12<3:32:02,  3.71it/s, loss=0.316] The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension.
Epoch 2/3:  13%|█▎        | 6724/51849 [2:19:20<23:53:47,  1.91s/it, loss=0.08]  The channel dimension is ambiguous. Got image

Epoch 2 Loss: 0.1491
Saved adapter to ./clip_lora_checkpoints/epoch-2


Epoch 3/3:   1%|          | 297/51849 [01:31<5:32:56,  2.58it/s, loss=0.2]    The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension.
Epoch 3/3:   1%|▏         | 734/51849 [03:50<6:03:33,  2.34it/s, loss=0.147] The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension.
The channel dimension is ambiguous. Got image shape (1, 1250, 3). Assuming channels are the first dimension.
Epoch 3/3:   6%|▌         | 2891/51849 [15:20<3:37:29,  3.75it/s, loss=0.146] The channel dimension is ambiguous. Got image shape (1, 1250, 3). Assuming channels are the first dimension.
The channel dimension is ambiguous. Got image shape (1, 1, 3). Assuming channels are the first dimension.
Epoch 3/3:   7%|▋         | 3793/51849 [20:06<3:24:59,  3.91it/s, loss=0.0182]The channel dimension is ambiguous. Got image s

Epoch 3 Loss: 0.1322
Saved adapter to ./clip_lora_checkpoints/epoch-3
Done!
